In [ ]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

load_dotenv()

API_KEY = os.getenv("DADATA_API_KEY")
URL = "https://suggestions.dadata.ru/suggestions/api/4_1/rs/suggest/car_brand"

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "Authorization": f"Token {API_KEY}",
}

brands = [
    "bmw",
    "audi",
    "toyota",
    "mercedes",
    "honda",
    "nissan",
    "lada",
    "ford",
    "chevrolet",
    "volvo"
]
rows = []

country_map = {
    "BMW": "Germany",
    "AUDI": "Germany",
    "TOYOTA": "Japan",
    "HONDA": "Japan",
    "NISSAN": "Japan",
    "MERCEDES": "Germany",
    "VAZ": "Russia",
    "FORD": "Usa",
    "CHEVROLET": "Usa",
    "VOLVO": "Sweden"

}

models_map = {
    "BMW": ["X5", "X3", "3 Series"],
    "AUDI": ["A4", "A6", "Q5"],
    "TOYOTA": ["Camry", "Corolla", "RAV4"],
    "HONDA": ["Civic", "Accord", "CR-V"],
    "NISSAN": ["Altima", "Qashqai", "X-Trail"],
    "MERCEDES": ["C-Class", "E-Class", "GLE"],
    "VAZ": ["Vesta", "Granta", "Niva"],
    "FORD": ["Focus", "Mustang", "Explorer"],
    "CHEVROLET": ["Malibu", "Camaro", "Tahoe"],
    "VOLVO": ["XC60", "XC90", "S60"]
}

year_map = {
    "BMW": 2020,
    "AUDI": 2021,
    "TOYOTA": 2023,
    "HONDA": 2019,
    "NISSAN": 2021,
    "MERCEDES": 2023,
    "VAZ": 2022,
    "FORD": 2020,
    "CHEVROLET": 2021,
    "VOLVO": 2023
}

for b in brands:
    response = requests.post(URL, headers=headers, json={"query": b})
    
    result = response.json()

    suggestions = result.get("suggestions", [])
    if suggestions:
        item = suggestions[0]

    
        d = item.get("data", {})
        logging.info(
    f"Loaded brand: {d.get('name')} | country: {country_map.get(d.get('id'))}"
)

        rows.append({
            "query": b,
            "value": item.get("value"),
            "id": d.get("id"),
            "name": d.get("name"),
            "name_ru": d.get("name_ru"),
        })
        
df = pd.DataFrame(rows)


df["country"] = df["id"].map(country_map) 
df["models"] = df["id"].map(models_map)
df["year"] = df["id"].map(year_map)

df = df.explode("models")
df = df.sort_values("country")

df.index = range(1, len(df) + 1)

df = df[["id", "name", "name_ru", "country", "models", "year"]]
df
logging.info(f"Dataset created with {len(df)} rows")
logging.info("Pipeline finished successfully ✅")
df.to_csv("car_brands.csv", index=False) 
df




2026-03-16 01:16:26,163 | INFO | Loaded brand: BMW | country: Germany
2026-03-16 01:16:26,230 | INFO | Loaded brand: Audi | country: Germany
2026-03-16 01:16:26,287 | INFO | Loaded brand: Toyota | country: Japan
2026-03-16 01:16:26,350 | INFO | Loaded brand: Mercedes-Benz | country: Germany
2026-03-16 01:16:26,400 | INFO | Loaded brand: Honda | country: Japan
2026-03-16 01:16:26,452 | INFO | Loaded brand: Nissan | country: Japan
2026-03-16 01:16:26,513 | INFO | Loaded brand: LADA (ВАЗ) | country: Russia
2026-03-16 01:16:26,574 | INFO | Loaded brand: Ford | country: Usa
2026-03-16 01:16:26,633 | INFO | Loaded brand: Chevrolet | country: Usa
2026-03-16 01:16:26,688 | INFO | Loaded brand: Volvo | country: Sweden
2026-03-16 01:16:26,702 | INFO | Dataset created with 30 rows
2026-03-16 01:16:26,703 | INFO | Pipeline finished successfully ✅


,id,name,name_ru,country,models,year
1,BMW,BMW,БМВ,Germany,X5,2020
2,BMW,BMW,БМВ,Germany,X3,2020
3,BMW,BMW,БМВ,Germany,3 Series,2020
4,AUDI,Audi,Ауди,Germany,A4,2021
5,AUDI,Audi,Ауди,Germany,A6,2021
6,AUDI,Audi,Ауди,Germany,Q5,2021
7,MERCEDES,Mercedes-Benz,Мерседес-Бенц,Germany,C-Class,2023
8,MERCEDES,Mercedes-Benz,Мерседес-Бенц,Germany,E-Class,2023
9,MERCEDES,Mercedes-Benz,Мерседес-Бенц,Germany,GLE,2023
10,NISSAN,Nissan,Ниссан,Japan,X-Trail,2021
